# **SNU AI Challenge code

## **Task**

주어진 스토리라인(캡션)에 맞게 4개의 이미지 프레임을 올바른 순서로 재배열하는 문제를 해결해야 합니다.

- **Input**: 자연어 문장과 여러 장의 프레임으로 구성된 데이터
  - 예: `{ "text": "자연어 문장", "frames": [image_3, image_1, image_4, image_2] }`
- **Output**: 정답 순서대로 다시 배열하였을 때 각 프레임의 위치
  - 예: `[3, 4, 1, 2]`
  - 정답 순서대로 다시 배열하였을 때 첫 번째 프레임은 3번째에 위치, 두 번째 프레임은 4번째에 위치, …

## **Overall Pipeline**

| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 (test.csv + 이미지) |
| Step 2 | 모델 로드 (Qwen2-VL-2B-Instruct from HuggingFace) |
| Step 3 | 추론 진행 (프롬프트 → 모델 생성 → 출력 파싱) |
| Step 4 | 결과 저장 (submission.csv) |


## 0. 환경 설정 및 패키지 설치


In [ ]:
%pip install -q torch torchvision transformers pandas pillow tqdm accelerate peft datasets
%pip install -q open_clip_torch
%pip install -q transformers accelerate qwen-vl-utils pillow pandas tqdm scikit-learn

In [ ]:
import os
import ast
import re
import gc
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

In [ ]:
#경로 설정
PROJECT_DIR = r"C:\SNU_2026"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_qwen3vl4b_enhanced.csv")

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR)
print("SUBMISSION_PATH:", SUBMISSION_PATH)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("test:", test_df.shape)

display(train_df.head())
display(test_df.head())

In [ ]:
#GPU 확인 및 모델 로드
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")

device = "cuda"

print("device:", device)
print("GPU:", torch.cuda.get_device_name(0))

gc.collect()
torch.cuda.empty_cache()

os.environ["HF_HUB_DISABLE_XET"] = "1"

MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("model loaded")
print("device:", next(model.parameters()).device)

In [ ]:
#정답 변환 함수
def parse_answer(answer):
    if isinstance(answer, str):
        return ast.literal_eval(answer)
    return answer


def order_to_submission_answer(order):
    """
    모델 출력: 시간순 이미지 번호
    예: [4, 2, 1, 3]

    제출 형식:
    Image 1이 몇 번째인지, Image 2가 몇 번째인지 ...
    예: [3, 2, 4, 1]
    """
    answer = [0] * 4

    for position, image_num in enumerate(order):
        answer[image_num - 1] = position + 1

    return answer


def parse_model_output(output_text):
    """
    모델 출력에서 [1, 2, 3, 4] 형태의 리스트만 추출합니다.
    이 값은 '시간순 이미지 번호'라고 가정합니다.
    """
    match = re.search(r"\[[^\]]+\]", output_text)

    if match is None:
        return [1, 2, 3, 4]

    try:
        pred_order = ast.literal_eval(match.group(0))
        pred_order = [int(x) for x in pred_order]

        if len(pred_order) == 4 and sorted(pred_order) == [1, 2, 3, 4]:
            return pred_order

    except Exception:
        pass

    return [1, 2, 3, 4]

In [ ]:
#강화 프롬프트
def make_qwen_messages(row, image_dir):
    img_files = [
        row["Input_1"],
        row["Input_2"],
        row["Input_3"],
        row["Input_4"],
    ]

    content = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, str(row["Id"]), str(img_file))

        content.append({
            "type": "image",
            "image": img_path,
        })

        content.append({
            "type": "text",
            "text": f"\nThis is Image {i + 1}.\n",
        })

    sentence = str(row["Sentence"])

    prompt = f"""
The 4 images above are SHUFFLED video frames.
They are NOT guaranteed to be in chronological order.

Sentence describing the event:
"{sentence}"

Your task:
Find the correct chronological order of Image 1, Image 2, Image 3, and Image 4.

Important:
- Do NOT assume the input image order is correct.
- Use only visual evidence from the images.
- Focus on what happens before and after.
- Compare hand movement, object movement, body pose, scene changes, and action progress.
- The answer must be the order of image numbers from earliest to latest.

Return ONLY a Python list of 4 integers.
Use each number 1, 2, 3, 4 exactly once.
Do not write any explanation.

Correct output format example:
[4, 2, 1, 3]
""".strip()

    content.append({
        "type": "text",
        "text": prompt,
    })

    return [
        {
            "role": "user",
            "content": content,
        }
    ]

In [ ]:
#예측함수
def predict_qwen_order(row, image_dir):
    messages = make_qwen_messages(row, image_dir)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    pred_order = parse_model_output(output_text)
    pred_answer = order_to_submission_answer(pred_order)

    return pred_order, pred_answer, output_text

In [ ]:
#validation 평가
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["Answer"],
)

valid_small = valid_part.head(200).reset_index(drop=True)

correct = 0
total = 0
identity_count = 0
bad_outputs = []

for idx, row in tqdm(valid_small.iterrows(), total=len(valid_small)):
    pred_order, pred_answer, raw = predict_qwen_order(row, TRAIN_IMAGE_DIR)
    true_answer = parse_answer(row["Answer"])

    if pred_answer == true_answer:
        correct += 1

    if pred_order == [1, 2, 3, 4]:
        identity_count += 1

    total += 1

    if sorted(pred_order) != [1, 2, 3, 4]:
        bad_outputs.append((row["Id"], raw))

accuracy = correct / total

print("valid_small accuracy:", accuracy)
print("identity ratio:", identity_count / total)
print("bad outputs:", len(bad_outputs))

In [ ]:
#Test 추론
predictions = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred_order, pred_answer, raw = predict_qwen_order(row, TEST_IMAGE_DIR)

    predictions.append({
        "Id": row["Id"],
        "Answer": str(pred_answer),
    })

submission_df = pd.DataFrame(predictions)

print("submission:", submission_df.shape)
display(submission_df.head())

## 4. 결과 저장 (Submission)

### 제출 파일 형식 (submission.csv)

제출 파일은 반드시 아래 형식을 따라야 합니다:

| 컬럼 | 타입 | 설명 | 예시 |
|------|------|------|------|
| `Id` | string | 테스트 샘플 고유 ID | `WI8o3F` |
| `Answer` | string | 정답 순서대로 다시 배열하였을 때 각 프레임의 위치 (Python 리스트 형태의 문자열) | `[3, 1, 4, 2]` |

#### 올바른 예시

```csv
Id,Answer
WI8o3F,[3, 1, 4, 2]
Ae7zLm,[1, 2, 3, 4]
...


In [ ]:
#제출 파일 저장
submission_df = test_df[["Id"]].merge(
    submission_df,
    on="Id",
    how="left",
)

submission_df["Answer"] = submission_df["Answer"].fillna(str([1, 2, 3, 4]))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH)
print("submission shape:", submission_df.shape)

display(submission_df.head())